# CatBoost-weighted three-model ensemble
Average probabilities across folds and seeds within each model. Use fixed weights of 80% CatBoost, 10% LightGBM and 10% XGBoost before applying the 0.5 threshold.

In [3]:
%pip install catboost lightgbm xgboost pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from catboost import CatBoostClassifier
from pathlib import Path
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir())  # project root
DATA = ROOT / 'data' / 'raw'
SUBMISSIONS = ROOT / 'submissions'
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
y = train.pop('Transported').astype(int)
rows = pd.concat([train, test], ignore_index=True)
assert sample.PassengerId.equals(test.PassengerId) and rows.PassengerId.is_unique

## Passenger and cabin features
The features use cabin occupancy, age and cabin bands, spending bands, and passenger groups. Group counts and unanimous group attributes use the complete unlabeled passenger manifest. Deck and side are used directly, not as deck transport rates, because rates calculated before splitting for validation would expose validation labels.

In [5]:
SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def features(rows):
    x = rows.copy()
    group = x.pop("PassengerId").str.split("_").str[0]
    family = x.pop("Name").str.split().str[-1]
    x[["Deck", "Number", "Side"]] = x.pop("Cabin").str.split("/", expand=True)
    x["Number"] = pd.to_numeric(x.Number)
    x["GroupNumber"] = group.astype(int)
    x["GroupSize"] = group.groupby(group).transform("size")
    x["FamilySize"] = family.groupby(family).transform("size")
    for col in ["Deck", "Side", "Number", "HomePlanet"]:
        matches = x[col].groupby(group)
        unanimous = matches.transform("first").where(matches.transform("nunique") == 1)
        x[col] = x[col].fillna(unanimous)
    x["TotalSpend"] = x[SPEND].sum(axis=1, min_count=1)
    x["MissingSpend"] = x[SPEND].isna().sum(axis=1)
    x["GroupMeanSpend"] = x.TotalSpend.groupby(group).transform("mean")
    x.loc[x.CryoSleep.eq(True), SPEND] = x.loc[x.CryoSleep.eq(True), SPEND].fillna(0)
    for col in ["Deck", "Side", "HomePlanet", "Destination", "CryoSleep", "VIP"]:
        x[col] = x[col].astype("string").fillna("Unknown").astype("category")
    return x

In [6]:
x = features(rows)
x['CabinMembers'] = rows.groupby('Cabin').PassengerId.transform('size')
for source, cuts in {'Age': [17, 43], 'Number': [300, 690, 1170],
                     'GroupNumber': [3400, 7300], 'TotalSpend': [1, 2250]}.items():
    x[source + 'Band'] = pd.cut(x[source], [-np.inf, *cuts, np.inf], labels=False)



In [7]:
folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(x.iloc[:len(y)], y))

In [8]:
inputs = pd.get_dummies(x, columns=x.select_dtypes('category').columns, dtype=float)
seeds = (42, 73, 101)

def make_model(name, seed):
    params = dict(n_estimators=550, learning_rate=.03, random_state=seed, n_jobs=4,
                  subsample=.85, colsample_bytree=.9)
    if name == 'LightGBM':
        return LGBMClassifier(**params, num_leaves=15, subsample_freq=1, verbosity=-1)
    return XGBClassifier(**params, max_depth=4, tree_method='hist')

def evaluate(name, seed):
    oof = np.full(len(y), np.nan)
    predictions = []
    for tr, va in folds:
        model = make_model(name, seed)
        model.fit(inputs.iloc[tr], y.iloc[tr])
        oof[va] = model.predict_proba(inputs.iloc[va])[:, 1]
        predictions.append(model.predict_proba(inputs.iloc[len(y):])[:, 1])
    assert np.isfinite(oof).all() and np.isfinite(predictions).all()
    print(name, seed, 'finished', flush=True)
    return oof, np.mean(predictions, axis=0)

In [9]:
validation = pd.DataFrame(index=y.index)
predictions = pd.DataFrame(index=test.index)
for name in ['LightGBM', 'XGBoost']:
    results = [evaluate(name, seed) for seed in seeds]
    validation[name + ' seed 42'] = results[0][0]
    validation[name] = np.mean([v for v, _ in results], axis=0)
    predictions[name] = np.mean([p for _, p in results], axis=0)


LightGBM 42 finished
LightGBM 73 finished
LightGBM 101 finished
XGBoost 42 finished
XGBoost 73 finished
XGBoost 101 finished


In [10]:
def evaluate_catboost(inputs, seed):
    predictions = np.full(len(y), np.nan)
    test_predictions = []
    for fold, (tr, va) in enumerate(folds, start=1):
        model = CatBoostClassifier(iterations=2000, depth=6, learning_rate=.01,
                    cat_features=inputs.select_dtypes('category').columns.tolist(),
                    thread_count=4, verbose=False, random_seed=seed, allow_writing_files=False)
        model.fit(inputs.iloc[tr], y.iloc[tr])
        predictions[va] = model.predict_proba(inputs.iloc[va])[:, 1]
        test_predictions.append(model.predict_proba(inputs.iloc[len(y):])[:, 1])
        print('Finished fold', fold, flush=True)
    assert np.isfinite(predictions).all()
    return predictions, np.mean(test_predictions, axis=0)

results = [evaluate_catboost(x, seed) for seed in (42, 73, 101)]
seed42_oof, seed42_test = results[0]
seed_oof = np.mean([oof for oof, _ in results], axis=0)
seed_test = np.mean([test for _, test in results], axis=0)

Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5


In [11]:
validation['CatBoost'] = seed_oof
predictions['CatBoost'] = seed_test
weights = pd.Series({'CatBoost': .8, 'LightGBM': .1, 'XGBoost': .1})
for frame in (validation, predictions):
    frame['Weighted'] = frame[weights.index].dot(weights)
correct = validation[['CatBoost', 'Weighted']].ge(.5).eq(y, axis=0)
print(correct.mean())
output = sample.copy()
output['Transported'] = predictions.Weighted.ge(.5)
path = SUBMISSIONS / 'submission_weighted_80_10_10.csv'
output.to_csv(path, index=False)
print(path, len(output), 'passengers')
saved = SUBMISSIONS / 'submission_catboost_seeds.csv'
if saved.exists():
    previous = pd.read_csv(saved)
    assert previous.PassengerId.equals(output.PassengerId)
    print('CatBoost reproduction differences:', (previous.Transported != predictions.CatBoost.ge(.5)).sum())
    print('Weighted predictions changed:', (previous.Transported != output.Transported).sum())

CatBoost    0.813068
Weighted    0.813873
dtype: float64
c:\Users\shaya\titanic-spaceShip\submissions\submission_weighted_80_10_10.csv 4277 passengers
CatBoost reproduction differences: 0
Weighted predictions changed: 38
